the overall priority system should be something like:
get nortons.
if fewer than 5 nortons, get sbns
if fewer than 5 sbsn, get strict parens
if fewer than 5 strict parens, get loose parens

#that requires an update to the classes to store strict and loose parens collection groups.


return all citations. 

In [ ]:
from single_file_processing_functions import convert_pdf_to_no_ws_txt as pdf_converter

In [ ]:
strawson_chapter = pdf_converter('pdfs/SCHIHA-3.2.pdf')

In [ ]:
from single_file_processing_classes import *

In [ ]:
import re

In [ ]:
new_norton = """
                        Abs(tract)*\.*§*\d+([-–—,]\d{1,2})*|
                        App(endix)*\.*§*\d+([-–—,]\d{1,2})*|
                        ((I{1,3}|[123])\.)(([i]{1,3}|IV|[I]{1,3}|[1-4])\.)(\d{1,2})(\.[1-9]\d{0,2}([-–—]\d{1,2}){0,1}){0,1}""", re.X

In [ ]:
class Paper:
    def __init__(self, txt_file_name):
        self.name = txt_file_name
        self.nortonCites = []
        self.sbnCites = []
        self.rawParenthesesCapture = []
        self.tCites = []
        self.pageNumCites = []
        self.rawNortonScore = {}
        self.rawSbnScore = {}
        self.rawTScore = {}
        self.rawPageNumScore = {}
        self.a_l_p = {}
        self.a_w_p = {}
        self.s_l_p = {}
        self.s_w_p = {}
        self.totalStrictCites = 0
        self.totalAggressiveCites = 0
        self.s_w_c = {}
        self.s_l_c = {}
        self.a_w_c = {}
        self.a_l_c = {}
        self.biblio = {}
        

    def NortonSearch(self):
        citationCounter = 0 #citation counter to be used in numbering citations
        #create the citation search object
        #search pattern is
##        Capture Abstract cites with optional dash additions
##        Capture Appendix cites with optional dash additions
##        Capture Main body cites as follows:
##            Book. (capture both roman and arabic numeral versions)
##            Part. (capture both roman (capitalized and not) and arabic numerals)
##            Section (I left of the '.' here to be able to capture citations that are only Book.Part.Section with no paragraph citation)
##            optional .Paragraph(s with optional dash separator for a range of paragraphs
        nortonPattern = re.compile("""  Abs\.*(tract)*§*\d+([-–—,]\d{1,2})*|
                                        App\.*(endix)*§*\d+([-–—,]\d{1,2})*|
                                        ((I{1,3}|[123]))
                                        (\.([i]{1,3}|IV|[I]{1,3}|[1-4]))
                                        (\.\d{1,2})
                                        (
                                            (\.)(?=\d{1,2})
                                            \d{1,2}(?!\d)
                                            ([-–—,]\d{1,2}(?!([\d]|(\.\d))))*
                                        )*""", flags=re.X|re.I)        
        #make the text of the paper accessible and generate the match objects
        text_name = 'txts/'+self.name+'.txt'
        paper_to_search = open(text_name, "r")
        text_to_search = paper_to_search.read().strip().replace(' ','')
        paper_to_search.close()
        matchObjects = nortonPattern.finditer(text_to_search)
        #create a citation for each match object
        for match in matchObjects:
            citationCounter +=1
            citationObject = Citation(self.name, citationCounter, match.group())
            citationObject.startPoint = match.start()
            citationObject.endPoint = match.end()
            citationObject.search_term = match.re
            self.nortonCites.append(citationObject)
        if len(self.nortonCites) == 0:
            pass

    def SbnSearch(self):
        citationCounter = 0 #citation counter to be used in numbering citations
        #create the citation search object
##        the search pattern this time is to start with SBN
##        then cover page numbers (I got rid of 0 as a starting point because a file happened to have a weird SBN0 followed by a long string of numbers
##        next I have an optional dash and comma separator that can be repeated to capture the multiple pages and ranges that get cited
##        this will require some cleaning because sometimes you get a random 'i' following the comma
        sbnPattern = re.compile(""" (?<!I)
                                    (SBN)
                                    ([1-9]\d+|[xvi]+|[XVI]+)
                                    ([-–—,](\d+|[xvi]+|[XVI]+))*""", re.X)
        #make the text of the paper accessible and generate the match objects
        text_name = 'txts/'+self.name+'.txt'
        paper_to_search = open(text_name, "r")
        text_to_search = paper_to_search.read().strip().replace(' ','')
        paper_to_search.close()
        matchObjects = sbnPattern.finditer(text_to_search)
        #create a citation for each match object
        for match in matchObjects:
            citationCounter +=1
            citationObject = Citation(self.name, citationCounter, match.group())
            citationObject.startPoint = match.start()
            citationObject.endPoint = match.end()
            citationObject.search_term = match.re
            self.sbnCites.append(citationObject)
        if len(self.sbnCites) == 0:
            pass

    def parenthesesCapture(self):
        citationCounter = 0 #citation counter to be used in numbering citations
        #create the citation search object
        #This idea behind this search string is to get anything in parentheses with the following structure:
            #First, it can optionally start with either a 'T', 'THN', 'Treatise', or 'Hume'
            #Second, there can be a run of some intervening text but not a close parens or any numbers
            #Third, we get a page number citation with an optional p, p., or pp.
            #fourth, we get up to a three digit page number or range of up to 3 digit page numbers in either
                #roman or arabic numerals
            #The only thing I haven't figured out how to capture yet is a brief comment that appears in a few
            #cases where the authro says something like, 'my emphases' or 'italics mine'. I think that might
            #require a different search with a more restrictive start to the parentheses
        pattern = re.compile('\((T|THN|Treatise|Hume)*([A-Z]|[a-z]|[,.])*(p*\.{0,1}(\d{1,3}|[xvi]+|[XVI]+)([-–—,](\d+|[xvi]{1,5}|[XVI]{1,5}))*)\)')
        #make the text of the paper accessible and generate the match objects
        text_name = 'txts/'+self.name+'.txt'
        paper_to_search = open(text_name, "r")
        text_to_search = paper_to_search.read().strip().replace(' ','')
        paper_to_search.close()
        matchObjects = pattern.finditer(text_to_search)
        #create a citation for each match object
        for match in matchObjects:
            citationCounter +=1
            citationObject = Citation(self.name, citationCounter, match.group())
            citationObject.startPoint = match.start()
            citationObject.endPoint = match.end()
            citationObject.search_term = match.re
            self.rawParenthesesCapture.append(citationObject)
        if len(self.rawParenthesesCapture) == 0:
            pass
        
    
    def a_l_p_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-a-l-p.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.a_l_p.items():
            csv_writer.writerow(pair)
    
    def a_w_p_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-a-w-p.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.a_w_p.items():
            csv_writer.writerow(pair)
    
    def s_l_p_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-s-l-p.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.s_l_p.items():
            csv_writer.writerow(pair)
    
    def s_w_p_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-s-w-p.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.s_w_p.items():
            csv_writer.writerow(pair)
    
    def a_l_c_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-a-l-c.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.a_l_c.items():
            csv_writer.writerow(pair)
    
    def a_w_c_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-a-w-c.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.a_w_c.items():
            csv_writer.writerow(pair)
    
    def s_l_c_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-s-l-c.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.s_l_c.items():
            csv_writer.writerow(pair)
    
    def s_w_c_CSV(self):
        csv_file = open('csvs/'+self.name[5:-4]+'-s-w-c.csv', 'w')
        csv_writer = csv.writer(csv_file)
        for pair in self.s_w_c.items():
            csv_writer.writerow(pair)
    
    def csvScoreDump(self):
        self.a_l_p_CSV()
        self.a_w_p_CSV()
        self.s_l_p_CSV()
        self.s_w_p_CSV()
        self.a_l_c_CSV()
        self.a_w_c_CSV()
        self.s_l_c_CSV()
        self.s_w_c_CSV()
        print('csvs dumped')


In [ ]:
import re
from master_function_list import abstractCleaner, dotAppAdder

In [ ]:
from master_function_list import ultimateParser as uP

Today, then, I've cleaned up a few of the new PDF files—Thiel book, Strawson book, two articles from Strawson and Noonan in the same companion book, the MacIntyre file downloaded May 22, rocknack book, kail chapter from book, new scherwain article...basically all the new pdfs i've acquired. I need to go back to double check some previous files to see whether they were processing from pd correctly

I've also added the ability to get the § sign parsed properly (this is already in the master function list.


I've also fixed some of the search strings in the class objects.

i still need to add teh more and less aggressive search options to the class definitions

Another search idea is to look for parentheses that start with 'Hume' and have a three digit page number in them, but are not four digit nubmers. That would capture some more citations than I'm getting now and would be very safe. So, inside a parentheses, the parentheses starts with hume and is a 1-3 digit number. Here's a regex to get 1-3 digit numbers only: '(?<!\d)\d{1,3}(?!\d)'. So the safest are Parens that start with Hume or Treatise or T followed by a page number, then we get parens that start with Hume or Treatise and have an up to 3 digit number inside the parentheses. 

In [ ]:
def processPaper(paper_obj):
    #this function take a paper object in and will fill the search lists with citations
    #First, conduct the searches
    paper_obj.NortonSearch()
    paper_obj.SbnSearch()
    paper_obj.parenthesesCapture()
    
    #second, set the easy .cleanedCitations
    for cite in paper_obj.nortonCites:
        cite.cleanedCitation = cite.rawCitationText
    for cite in paper_obj.sbnCites:
        cite.cleanedCitation = cite.rawCitationText[3:]
    
    #third, sort the parentheses
    #in the sorting, we'll do a first pass to get the citations that start with T or Hume or THN or Treatise:
    strict_check = re.compile(r'(Hume|Treatise|THN|T)(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))+', re.I)
    num_check = re.compile(r'(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))+', re.I)
    for cite in paper_obj.rawParenthesesCapture:
        #make the strict check:
        #I start it at the first character to ignore the open parens that
        if strict_check.match(cite.rawCitationText[1:]) != None:
            if num_check.search(cite.rawCitationText) != None:
                num_text = num_check.search(cite.rawCitationText).group().replace(' ', '')
                #remove 'p', 'P', '.' from the nubmers
                num_text = num_text.replace('p','')
                num_text = num_text.replace('P','')
                num_text = num_text.replace('.','')
                #set the cleanedCitation to the num_text
                cite.cleanedCitation = num_text
                #add the cite to the tCites
                paper_obj.tCites.append(cite)
        else:
            if num_check.search(cite.rawCitationText) != None:
                num_text = num_check.search(cite.rawCitationText).group().replace(' ', '')
                #remove 'p', 'P', '.' from the nubmers
                num_text = num_text.replace('p','')
                num_text = num_text.replace('P','')
                num_text = num_text.replace('.','')
                #set the cleanedCitation to the num_text
                cite.cleanedCitation = num_text
                #add the cite to pageNumCites, if it's more than one digit
                #because otherwise we catch too much argument numbering use
                #of parentheses:
                if len(num_text) > 1:
                    paper_obj.pageNumCites.append(cite)

    print (paper_obj.name, 'processed')

In [ ]:
strict_check = re.compile(r'(Hume|Treatise|THN|T)(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))+', re.I)
test_text = '(Hume, p.25 - pp.300 )'
print(strict_check.search(test_text).group())
num_check = re.compile(r'((?<!\d)\d{1,3}(?!\d))(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))*', re.I)
print(num_check.search(test_text).group())
print(num_check.search(test_text).group().replace(' ', '').replace('p','').replace('P','').replace('.',''))


In [ ]:
aggressive_check = re.compile(r'(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))+', re.I)
aggressive_check.search(test_text).group()

So, What I have now I can take in a text file and populate all of .cleanedCitation fields in every citation type. Next, I want to come up with a smarter score generator. 

The way the score generator will work is:
1. Create a score card dictionary
2. Calculate the Norton scores for each norton cite
3. If there are under 5 norton citations, go get the SBNs
4. If there are under 5 SBNs go get the strict Ts
5. If there are under 5 strict Ts, go get all the parens.

In [ ]:
def chapterScores(para_scores):
    #this function takes in a dictionary score sheet for paragraph scores
    #and returns a dictionary score sheet for chapter scores
    chapter_scores = {}
    chapter_pattern = re.compile(r'(Abs)|(App\.)|(0)|(\d\.\d\.\d{1,2})')
    for para in para_scores.keys():
        chapter = chapter_pattern.match(para).group()
        if chapter not in chapter_scores.keys():
            chapter_scores[chapter] = 0
        chapter_scores[chapter] += para_scores[para]
    return chapter_scores



def citesToScores(cite_collection):
    #this function takes in a list of cites as you'd have in a paper object that's been processed
    #and returns a dictionary object with a count of total successful cites that were scored
    #and a scoresheet dictionary at the paragraph level
    out_sheet = {}
    successful_cites = 0
    for para in treatise_paragraph_list:
        out_sheet[para] = 0
    for cite in cite_collection:
        try:
            for pair in uP(cite.cleanedCitation):
                out_sheet[pair[0]] += pair[1]
            successful_cites += 1
        except Exception as e:
            print(cite.cleanedCitation, 'generated an error while scoring:', e)
    
    do = {}
    do['score_sheet'] = out_sheet
    do['total_cites'] = successful_cites
            
    return do

def relativeScoreCalc(score_dict):
    #this function calculates a score sheet relative to the paper object that is processed
    #ie how what percentage of the article's total cites go to a particular paragraph
    #get the total number of points awarded in the score_dict
    total_points = 0
    for entry in score_dict.keys():
        total_points += score_dict[entry]

    if total_points == 0:
        return score_dict
    
    else:
        #create the blank relative sheet
        relative_sheet = {}
        for para in treatise_paragraph_list:
            relative_sheet[para] = 0
        #fill in the blank relative sheet
        for para in relative_sheet.keys():
            relative_sheet[para] = 100 * score_dict[para]/total_points
        
        return relative_sheet
      

def paperScoreSetter(paper_obj):
    paper_obj.rawNortonScore = citesToScores(paper_obj.nortonCites)['score_sheet']
    paper_obj.rawSbnScore = citesToScores(paper_obj.sbnCites)['score_sheet']
    paper_obj.rawTScore = citesToScores(paper_obj.tCites)['score_sheet']
    paper_obj.rawPageNumScore = citesToScores(paper_obj.pageNumCites)['score_sheet']
    
    if len(paper_obj.nortonCites) > 5:
        paper_obj.s_w_p = paper_obj.rawNortonScore
        paper_obj.totalStrictCites = citesToScores(paper_obj.nortonCites)['total_cites']
         
    else:
        if len(paper_obj.sbnCites) > 5:
            strict_cites = paper_obj.nortonCites + paper_obj.sbnCites
            paper_obj.s_w_p = citesToScores(strict_cites)['score_sheet']
            paper_obj.totalStrictCites = citesToScores(strict_cites)['total_cites']
        else:
            strict_cites = paper_obj.nortonCites + paper_obj.sbnCites + paper_obj.tCites
            paper_obj.s_w_p = citesToScores(strict_cites)['score_sheet']
            paper_obj.totalStrictCites = citesToScores(strict_cites)['total_cites']

    agg_cites = paper_obj.nortonCites + paper_obj.sbnCites + paper_obj.tCites + paper_obj.pageNumCites
    paper_obj.totalAggressiveCites = citesToScores(agg_cites)['total_cites']
    paper_obj.a_w_p = citesToScores(agg_cites)['score_sheet']
    
    paper_obj.s_l_p = relativeScoreCalc(paper_obj.s_w_p)
    paper_obj.a_l_p = relativeScoreCalc(paper_obj.a_w_p)
    
    paper_obj.s_w_c = chapterScores(paper_obj.s_w_p)
    paper_obj.s_l_c = chapterScores(paper_obj.s_l_p) 
    paper_obj.a_w_c = chapterScores(paper_obj.a_w_p)
    paper_obj.a_l_c = chapterScores(paper_obj.a_l_p)
    
    print('finished setting scores for', paper_obj.name)

So, in addition to the previous steps forward I have now:
- updated the Paper Class ot have the right attributes
- written a function to set the raw and relative score sheets to distinguish smart and relative
- created the relevant homes for citations so that each type of search has its own citations and score sheets.
- can dump all scores into csvs.

What I need to work on now:
- I can create paper objects based on a CSV that's an output from the PhilPapers data base
- For those paper objects, I can process them to get a set of strict cites and aggressive cites, including a total count of successfully processed citations. I also have each set of citations that's attached to each of my four citation patterns (norton, sbn, t, and appropriate length page number)
- The next step will be to create to and from dictionary functions so that I can move back and forth between database stored objects and a website.
- i want to be able to generate on deman comparisons for relative scores, strict and aggressive, while also controlling for how many successive cites for each
- i also need to go back through the existing txt files to make sure I don't have too many (cid:xxx) poluted files

In [ ]:
test_paper.csvScoreDump()

I also need to use the base here to update the main files.

I should try to run everything into a full single-file processor that will read a text, create all the different scores, and output the csvs.

As of the end of Friday, I updated some of the search parameters, added another two papers and started compiling chapters as they come in from ILL for Penelhum's Themes from Hume book;

still need the to and from dictionary functions to facilitate writing and reading from the gcloud database

also need to process all the docs with the new smart processor and then start getting the comparasion sets up to date.

Now let's work on a Chapter Conversion function that will work by adding up the citations from a score sheet to yield chapter scores.

In [ ]:
chapter_pattern = re.compile(r'(Abs)|(App\.)|(0)|(\d\.\d\.\d{1,2})')
test_dict = second_test.relativeStrictScore
test_ch_dict = {}
for para in test_dict.keys():
    chap_pat = chapter_pattern.match(para).group()
    if chap_pat not in test_ch_dict.keys():
        test_ch_dict[chap_pat] = 0
    test_ch_dict[chap_pat] += test_dict[para]

relative_dict_total = 0
for para in test_dict.keys():
    relative_dict_total += test_dict[para]
print('the total input dict scores add to', str(relative_dict_total))

chap_total = 0
for chap in test_ch_dict.keys():
    chap_total += test_ch_dict[chap]
print('the total chapter dict scores add to', str(chap_total))

for chap in test_ch_dict.keys():
    print(chap, test_ch_dict[chap])

In [ ]:
#Chapters: Appendix, Abstract, Book.Section.Chapter
#The way to do this is to start with any dictionary score sheet as input and return a chapter score sheet
#So, you put a paragraph based score sheet dictionary in and get out a chapter based score sheet


def chapterScores(para_scores):
    chapter_scores = {}
    chapter_pattern = re.compile(r'(Abs)|(App\.)|(0)|(\d\.\d\.\d{1,2})')
    for para in para_scores.keys():
        chapter = chapter_pattern.match(para).group()
        if chapter not in chapter_scores.keys():
            chapter_scores[chapter] = 0
        chapter_scores[chapter] += para_scores[para]
    return chapter_scores

In [ ]:
for chapter in chapterScores(second_test.relativeStrictScore).keys():
    print(chapter, chapterScores(second_test.relativeStrictScore)[chapter])

Now that the chapter based score calculator is integrated we can update the Paper objects with Chapter based scores. Maybe in terms of the built in scores, I'll just attach this to aggressive and strict raw and relative dictionaries as attributes of the Paper object. That will then let us do chapter and paragraph based similarity comparisons.

In [ ]:
total_counter = 0
for key in third_paper.strictRelativeChapterScore.keys():
    if third_paper.strictRelativeChapterScore[key] > 0:
        total_counter += third_paper.strictRelativeChapterScore[key]
        print(key, third_paper.strictRelativeChapterScore[key])
print(total_counter)

In [ ]:
third_paper.rawStrictScore['0.1']

In [ ]:
import os
#make master score list
master_strict_para_list = {}
for para in treatise_paragraph_list:
    master_strict_para_list[para] = 0 
    
master_strict_chap_list = chapterScores(master_strict_para_list)

txt_dir = os.listdir('txts')

In [ ]:
for filename in txt_dir:
    if 'txt' in filename:
        print('working on', filename)
        paper = newPaper('txts/'+filename)
        processPaper(paper)
        paperScoreSetter(paper)
        for para in master_strict_para_list.keys():
            master_strict_para_list[para] += paper.rawStrictScore[para]
        for chapter in master_strict_chap_list.keys():
            master_strict_chap_list[chapter] += paper.strictRawChapterScore[chapter]
        if paper.totalStrictCites < 1:
            print(filename, ": ", str(paper.totalStrictCites),file=zero_counter)
        print('done with', filename)
zero_counter.close()

In [ ]:
total_cites = 0
for para in master_strict_para_list:
    if master_strict_para_list[para] > 0:
        print(para, master_strict_para_list[para])
        total_cites += master_strict_para_list[para]
print(total_cites)        

In [ ]:
test_chap = chapterScores(master_strict_para_list)
total_cites = 0
for chap in test_chap.keys():
    if test_chap[chap] > 0:
        print(chap, test_chap[chap])
        total_cites += test_chap[chap]
print(total_cites)        

In [ ]:
import pandas as pd

In [ ]:
pdata = pd.DataFrame.from_dict(master_strict_para_list, orient='index', columns=['score'])
pdata['score'].sum()

In [ ]:
import matplotlib as plt

In [ ]:
pdata.plot.bar(figsize=(15,8)).spines['right'].set_color('none')

In [ ]:
cdata = pd.DataFrame.from_dict(master_strict_chap_list, orient='index', columns=['score'])
cdata['score'].sum()

In [ ]:
type(cdata.plot.bar(figsize=(15,8)))

In [ ]:
pdata['score'].sort_values()[-10:]

In [ ]:
cdata['score'].sort_values()[-10:]

In [ ]:
appendix = pdata['App.1':'Absi']

In [ ]:
appendix.plot.bar()

In [ ]:
b14 = pdata['1.4.6.1':'1.4.7.1'][:-1]

In [ ]:
b14.plot.bar()

In [ ]:
master_agg_para_list = {}
for para in treatise_paragraph_list:
    master_agg_para_list[para] = 0 
    
master_agg_chap_list = chapterScores(master_agg_para_list)

In [ ]:
for filename in txt_dir:
    if 'txt' in filename:
        print('working on', filename)
        paper = newPaper('txts/'+filename)
        processPaper(paper)
        paperScoreSetter(paper)
        for para in master_agg_para_list.keys():
            master_agg_para_list[para] += paper.rawAggressiveScore[para]
        for chapter in master_agg_chap_list.keys():
            master_agg_chap_list[chapter] += paper.aggressiveRawChapterScore[chapter]
        print('done with', filename)

In [ ]:
apdata = pd.DataFrame.from_dict(master_agg_para_list, orient='index', columns=['score'])

In [ ]:
apdata['score'].sum()

In [ ]:
apdata['score'].sort_values()

In [ ]:
apdata.plot.bar(figsize=(15,8))

In [ ]:
acdata = pd.DataFrame.from_dict(master_agg_chap_list, orient='index', columns=['score'])
acdata['score'].sum()

In [ ]:
acdata.plot.bar(figsize=(15,8))

In [ ]:
#need a double digit check to get rid of the single digit argument numbering stuff that happens inside of parentheses.
#to do that, we need a 'more than one digit check':

#here's the basic num_check:
num_check = re.compile(r'(([, \-\–\—p\.]*)((?<!\d)\d{1,3}(?!\d)))+', re.I)
#now i want to make sure that we've got a two digit number at least.
#i think the best way to do that is to just add to pageNumCites only if the group match is len greater than 1:

#doing that greatly reduced the number of credits to 1.1.1.1 and took it off the top spot of the paragraph contributions.
#so i think that's the right choice.
#now let's see how different the strict and aggressive results are.

In [ ]:
pdata['aggressive-score'] = apdata['score']

In [ ]:
cdata['aggressive-score'] = acdata['score']

In [ ]:
cdata['difference'] = abs(cdata['score']-cdata['aggressive-score'])

In [ ]:
pdata['difference'] = abs(pdata['score']-pdata['aggressive-score'])

In [ ]:
pdata

In [ ]:
cdata['difference'].sum()

In [ ]:
pdata['difference'].sum()

In [ ]:
#now to do this for relative cites. I'll do it in one pass

rel_agg_para_scores = relativeScoreCalc(master_strict_para_list)
rel_strict_para_scores = relativeScoreCalc(master_agg_para_list)
rel_agg_chap_scores = chapterScores(rel_agg_para_scores)
rel_strict_chap_scores = chapterScores(rel_strict_para_scores)

In [ ]:
rel_p_data = pd.DataFrame.from_dict(rel_strict_para_scores, orient='index', columns=['strict-score'])
rel_ap_data = pd.DataFrame.from_dict(rel_agg_para_scores, orient='index', columns=['aggressive-score'])
rel_p_data['aggressive-score'] = rel_ap_data['aggressive-score']
rel_p_data['difference'] = abs(rel_p_data['strict-score'] - rel_p_data['aggressive-score'])
print(rel_p_data['strict-score'].sum())
print(rel_p_data['aggressive-score'].sum())
rel_p_data.sort_values('difference')[-20:]  

In [ ]:
rel_c_data = pd.DataFrame.from_dict(rel_strict_chap_scores, orient='index', columns=['strict-score'])
rel_ac_data = pd.DataFrame.from_dict(rel_agg_chap_scores, orient='index', columns=['aggressive-score'])
rel_c_data['aggressive-score'] = rel_ac_data['aggressive-score']
rel_c_data['difference'] = abs(rel_c_data['strict-score'] - rel_c_data['aggressive-score'])
print(rel_c_data['strict-score'].sum())
print(rel_c_data['aggressive-score'].sum())

rel_c_data.sort_values('difference')[-20:]                          

In [ ]:
total_diff_chap = rel_c_data['difference'].sum()
total_diff_chap

In [ ]:
total_diff_para = rel_p_data['difference'].sum()
total_diff_para

In [ ]:
#The results there are pretty reasonable! Differences at the paragraph level are a bit washed out 
#at the chapter level.
#it also makes sense that the biggest differences are at the paragraphs that have the most citation generally


What I want to do now is have the program loop over every article, create 8 csvs for each file—there are three dimensions each of which has two values:
strict-aggressive
raw-relative
paragraph-chapter

the basic distinction is strict v aggressive. if you calculate the raw paragraph score for each of those, you can then derive the relative and chapter scores.

But all of those are calculated by the ScoreSetter for individual articles.

For literature level data, though, the chapter and relative data can be derived by first looping over all the strict and aggressive raw paragraph data.

for cleaning these up I need a canonical way to write out these dictionary names so that it's easy to remember.

strict/agg_raw/relative_para/chapter
is probably the way to go.



In [ ]:
#Working through literature level data file by file
#1. Get a text file from list of text files
import os
import csv
txts = os.listdir('txts')

#creating dictionaries to fill later, s_w_p and a_w_p, we'll use these later to create the
#lit level relative and chapter score sheets
lit_s_w_p = {}
for para in treatise_paragraph_list:
    lit_s_w_p[para] = 0 

lit_a_w_p = {}
for para in treatise_paragraph_list:
    lit_a_w_p[para] = 0 

#2. Create a paper object from the paper
for file in txts:
    if 'txt' in file:
        paper = newPaper('txts/'+file)
        #3. Process the paper processPaper(paper_obj)
        processPaper(paper)
        #4. Set thes scores for the paper object paperScoreSetter(paper_obj)
        paperScoreSetter(paper)
        #5. dump 8 csvs for the scores. fliename_strict/agg_raw/relative_para/chapter.csv file format
        paper.csvScoreDump()
        # Now I need to update the lit_s_w_p and lit_a_w_p dictionaries
        for para in paper.s_w_p.keys():
            lit_s_w_p[para] += paper.s_w_p[para]
        for para in paper.a_w_p.keys():
            lit_a_w_p[para] += paper.a_w_p[para]

def d_to_csv(dictionary, filename):
    file = open('csvs/'+filename+'.csv', 'w')
    csv_writer = csv.writer(file)
    for item in dictionary.items():
        csv_writer.writerow(item)
    file.close()

#dump the lit_s_w_p to csv
d_to_csv(lit_s_w_p, 'lit_s_w_p')
#7. create strict_relative_paragraph data, dump to csv
lit_s_l_p = relativeScoreCalc(lit_s_w_p)
d_to_csv(lit_s_l_p, 'lit_s_l_p')
#8. create strict_raw_chapter data, dump to csv 
lit_s_w_c = chapterScores(lit_s_w_p)
d_to_csv(lit_s_w_c, 'lit_s_w_c')
#9. create strict_relative_chapter data, dump to csv 
lit_s_l_c = chapterScores(lit_s_l_p)
d_to_csv(lit_s_l_c, 'lit_s_l_c')


#dump aggressive_raw_paragraph data to csv 
d_to_csv(lit_a_w_p, 'lit_a_w_p')
#11. From 10, create aggressive_relative_paragraph data, dump to csv 
lit_a_l_p = relativeScoreCalc(lit_a_w_p)
d_to_csv(lit_a_l_p, 'lit_a_l_p')
#12. From 10, create aggressive_raw_chapter data, dump to csv  
lit_a_w_c = chapterScores(lit_a_w_p)
d_to_csv(lit_a_w_c, 'lit_a_w_c')
#13. From 12, create aggressive_relative_chapter data, dump to csv 
lit_a_l_c = chapterScores(lit_a_l_p)
d_to_csv(lit_a_l_c, 'lit_a_l_c')
print('done generating data')

In [ ]:
#generating similarity data
#Let's start with relative data only. That gives us four sets to work with
#1. create a data frame from literature level strict_relative_paragraph data csv
#for each file in the list of text files, create a data frame from that file's strict_relative_paragraph data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, comparison_strict_relative_paragraph_data.csv
#2. create a data frame from literature level aggressive_relative_paragraph data csv
#for each file in the list of text files, create a data frame from that file's aggresive_relative_paragraph data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, comparison_aggressive_relative_paragraph_data.csv
#3. create a data frame from literature level strict_relative_chapter data data csv
#for each file in the list of text files, create a data frame from that file's strict_relative_chapter data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, strict_relative_chapter_paragraph_data.csv
#4. create a data frame from literature level aggressive_relative_chapter data data csv
#for each file in the list of text files, create a data frame from that file's aggressive_relative_chapter data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, aggressive_relative_chapter_paragraph_data.csv


In [ ]:
#1. create a data frame from literature level strict_relative_paragraph data csv
s_l_p = pd.read_csv('csvs/lit_s_l_p.csv', index_col=0, names=['Literature'])

In [ ]:
#for each file in the list of text files:
for file in txts:
    if 'txt' in file:
        csv_base_name = 'csvs/'+file[:-4]
        csv_s_l_p_name = csv_base_name+'-s-l-p.csv'
        #create a data frame from that file's strict_relative_paragraph data csv
        article_df = pd.read_csv(csv_s_l_p_name, index_col=0, names=[file[:-4]])
        #add that score column to the literature level data
        s_l_p[file[:-4]] = article_df[file[:-4]]

In [ ]:
#dump this literature level comparative data to a csv, comparison_strict_relative_paragraph_data.csv
s_l_p.to_csv('s_l_p_data.csv')

In [ ]:
#2. create a data frame from literature level aggressive_relative_paragraph data csv
#for each file in the list of text files, create a data frame from that file's aggresive_relative_paragraph data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, comparison_aggressive_relative_paragraph_data.csv
a_l_p = pd.read_csv('csvs/lit_a_l_p.csv', index_col=0, names=['Literature'])
for file in txts:
    if 'txt' in file:
        csv_base_name = 'csvs/'+file[:-4]
        csv_a_l_p_name = csv_base_name+'-a-l-p.csv'
        #create a data frame from that file's strict_relative_paragraph data csv
        article_df = pd.read_csv(csv_a_l_p_name, index_col=0, names=[file[:-4]])
        #add that score column to the literature level data
        a_l_p[file[:-4]] = article_df[file[:-4]]
a_l_p.to_csv('a_l_p_data.csv')

In [ ]:
#3. create a data frame from literature level strict_relative_chapter data data csv
#for each file in the list of text files, create a data frame from that file's strict_relative_chapter data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, strict_relative_chapter_paragraph_data.csv
s_l_c = pd.read_csv('csvs/lit_s_l_c.csv', index_col=0, names=['Literature'])
for file in txts:
    if 'txt' in file:
        csv_base_name = 'csvs/'+file[:-4]
        csv_s_l_c_name = csv_base_name+'-s-l-c.csv'
        #create a data frame from that file's strict_relative_paragraph data csv
        article_df = pd.read_csv(csv_s_l_c_name, index_col=0, names=[file[:-4]])
        #add that score column to the literature level data
        s_l_c[file[:-4]] = article_df[file[:-4]]
s_l_c.to_csv('s_l_c_data.csv')

In [ ]:
#4. create a data frame from literature level aggressive_relative_chapter data data csv
#for each file in the list of text files, create a data frame from that file's aggressive_relative_chapter data csv
    #add that score column to the literature level data
    #dump this literature level comparative data to a csv, aggressive_relative_chapter_paragraph_data.csv
a_l_c = pd.read_csv('csvs/lit_a_l_c.csv', index_col=0, names=['Literature'])
for file in txts:
    if 'txt' in file:
        csv_base_name = 'csvs/'+file[:-4]
        csv_a_l_c_name = csv_base_name+'-a-l-c.csv'
        #create a data frame from that file's strict_relative_paragraph data csv
        article_df = pd.read_csv(csv_a_l_c_name, index_col=0, names=[file[:-4]])
        #add that score column to the literature level data
        a_l_c[file[:-4]] = article_df[file[:-4]]
a_l_c.to_csv('a_l_c_data.csv')

Okay, so I now have exported 4 csvs which have all the relative citation data for each aggressive/strict paragraph/chapter data set. Each CSV can be imported into a data frame and I can go back to the similarity notebook to generate the similarity scores.

In [ ]:
def df_col_similarity(df, colx, coly):
    total_diff = 0
    for para in df.index:
        total_diff += abs(df[colx][para] - df[coly][para])
    return total_diff

Let's start with s_l_c

In [ ]:
s_l_c_similarity = pd.DataFrame(index=s_l_c.columns, columns=s_l_c.columns)
for column in s_l_c_similarity.columns:
    for ind in s_l_c_similarity.index:
        s_l_c_similarity[column][ind] = df_col_similarity(s_l_c,column,ind)
        

In [ ]:
s_l_c_similarity.to_csv('similarity_scores/s_l_c_similarity.csv')

In [ ]:
a_l_c_similarity = pd.DataFrame(index=a_l_c.columns, columns=a_l_c.columns)
for column in a_l_c_similarity.columns:
    for ind in a_l_c_similarity.index:
        a_l_c_similarity[column][ind] = df_col_similarity(a_l_c,column,ind)
a_l_c_similarity.to_csv('similarity_scores/a_l_c_similarity.csv')

In [ ]:
s_l_p_similarity = pd.DataFrame(index=s_l_p.columns, columns=s_l_p.columns)
for column in s_l_p_similarity.columns:
    for ind in s_l_p_similarity.index:
        s_l_p_similarity[column][ind] = df_col_similarity(s_l_p,column,ind)
s_l_p_similarity.to_csv('similarity_scores/s_l_p_similarity.csv')

In [ ]:
a_l_p_similarity = pd.DataFrame(index=a_l_p.columns, columns=a_l_p.columns)
for column in a_l_p_similarity.columns:
    for ind in a_l_p_similarity.index:
        a_l_p_similarity[column][ind] = df_col_similarity(a_l_p,column,ind)
a_l_p_similarity.to_csv('similarity_scores/a_l_p_similarity.csv')

In [ ]:
def p_to_dict(paper):
    od = {}
    od['name'] = paper.name
    norton_cites_out = []
    for cite in paper.nortonCites:
        norton_cites_out.append(c_to_dict(cite))
    od['nortonCites'] = norton_cites_out
    sbn_cites_out = []
    for cite in paper.sbnCites:
        sbn_cites_out.append(c_to_dict(cite))
    od['sbnCites'] = sbn_cites_out
    raw_out = []
    for cite in paper.rawParenthesesCapture:
        raw_out.append(c_to_dict(cite))
    od['rawParenthesesCapture'] = raw_out
    t_out = []
    for cite in paper.tCites:
        t_out.append(c_to_dict(cite))
    od['tCites'] = t_out
    pageNumOut = []
    for cite in paper.pageNumCites:
        pageNumOut.append(c_to_dict(cite))
    od['pageNumCites'] = pageNumOut
    od['rawNortonScore'] = paper.rawNortonScore
    od['rawSbnScore'] = paper.rawSbnScore
    od['rawTScore'] = paper.rawTScore
    od['rawPageNumScore'] = paper.rawPageNumScore
    od['a_l_p'] = paper.a_l_p
    od['a_w_p'] = paper.a_w_p
    od['s_l_p'] = paper.s_l_p
    od['s_w_p'] = paper.s_w_p
    od['totalStrictCites'] = paper.totalStrictCites
    od['totalAggressiveCites'] = paper.totalAggressiveCites
    od['s_w_c'] = paper.s_w_c
    od['s_l_c'] = paper.s_l_c
    od['a_w_c'] = paper.a_w_c
    od['a_l_c'] = paper.a_l_c
    od['biblio'] = paper.biblio
    
    return od

def c_to_dict(cite):
    od = {}
    od['order'] = cite.order
    od['paper'] = cite.paper
    od['startPoint'] = cite.startPoint
    od['endPoint'] = cite.endPoint
    od['rawCitationText'] = cite.rawCitationText
    od['cleanedCitation'] = cite.cleanedCitation
    
    return od
    

In [ ]:
def p_from_dict(in_dict):
    paper = newPaper(in_dict['name'])
    for cite in in_dict['nortonCites']:
        paper.nortonCites.append(c_from_dict(cite))
    for cite in in_dict['sbnCites']:
        paper.sbnCites.append(c_from_dict(cite))
    for cite in in_dict['rawParenthesesCapture']:
        paper.rawParenthesesCapture.append(c_from_dict(cite))
    for cite in in_dict['tCites']:
        paper.tCites.append(c_from_dict(cite))
    for cite in in_dict['pageNumCites']:
        paper.pageNumCites.append(c_from_dict(cite))
    paper.rawNortonScore = in_dict['rawNortonScore']
    paper.rawSbnScore = in_dict['rawSbnScore']
    paper.rawTScore = in_dict['rawTScore']
    paper.rawPageNumScore = in_dict['rawPageNumScore']
    paper.totalStrictCites = in_dict['totalStrictCites']
    paper.totalAggressiveCites = in_dict['totalAggressiveCites']
    paper.a_l_p = in_dict['a_l_p']
    paper.a_w_p = in_dict['a_w_p']
    paper.a_l_c = in_dict['a_l_c']
    paper.a_w_c = in_dict['a_w_c']
    paper.s_l_p = in_dict['s_l_p']
    paper.s_w_p = in_dict['s_w_p']
    paper.s_l_c = in_dict['s_l_c']
    paper.s_w_c = in_dict['s_w_c']
    paper.biblio = in_dict['biblio']
    
    return paper


def c_from_dict(in_dict):
    cite = Citation(in_dict['paper'], in_dict['order'], in_dict['rawCitationText'])
    cite.startPoint = in_dict['startPoint']
    cite.endPoint = in_dict['endPoint']
    cite.cleanedCitation = in_dict['cleanedCitation']
    
    return cite

In [ ]:
cpaper = newPaper('533Cottrell')
processPaper(cpaper)
paperScoreSetter(cpaper)
cotd = p_to_dict(cpaper)

In [ ]:
cotd['s_w_p']['1.4.6.4']

In [ ]:
cotd['nortonCites'][5]['cleanedCitation']

In [ ]:
revpaper = p_from_dict(cotd)

In [ ]:
revpaper.nortonCites[5].cleanedCitation

In [ ]:
print(cpaper.s_w_p == revpaper.s_w_p)
print(cpaper.s_l_p == revpaper.s_l_p)
print(cpaper.s_w_c == revpaper.s_w_c)
print(cpaper.s_l_c == revpaper.s_l_c)
print(cpaper.a_w_p == revpaper.a_w_p)
print(cpaper.a_l_p == revpaper.a_l_p)
print(cpaper.a_w_c == revpaper.a_w_c)
print(cpaper.a_l_c == revpaper.a_l_c)

In [ ]:
cotd

In [ ]:
import json
file = open('533Cottrell.json', 'w')
json.dump(cotd, file)
file.close()

In [ ]:
jfile = open('533Cottrell.json', 'r')
jd = json.load(jfile)
jpaper = p_from_dict(jd)
jfile.close()

In [ ]:
import pandas as pd


In [ ]:
refs = pd.read_csv('List-of-Refs.csv', header=0)

In [ ]:
refs

In [ ]:
na_in_dl = refs['Downloaded?'].notna()
na_in_dl

In [ ]:
dl_no_na = refs[na_in_dl]
dl_no_na

In [ ]:
dl_yes = dl_no_na['Downloaded?'].str.contains('yes', case=False)
dl_files = dl_no_na[dl_yes]

In [ ]:
dl_files

In [ ]:
import os
raw_txt_list = os.listdir('txts')
clean_txt_list = []
for file in raw_txt_list:
    if 'txt' in file:
        clean_txt_list.append(file)
print(len(raw_txt_list))
print(len(clean_txt_list))

In [ ]:
filename_filter = dl_files['Filename'].str.contains('23598030')

In [ ]:
print(dl_files[filename_filter]['Title of Work'].values[0])

In [ ]:
for file in clean_txt_list:
    try:
        filename_filter = dl_files['Filename'].str.contains(file[:-4])
        author = dl_files[filename_filter]['Author'].values[0]
        print(file[:-4], author)
    except Exception as e:
        print(e)

In [ ]:
#so, all txt files can be linked back to their bibliographic information
#now i need to go the other way, try to find the missing txt files for allegedly downloaded files

In [ ]:
biblio_filenames = []
for filename in dl_files['Filename'].values:
    biblio_filenames.append(filename[:-4])
biblio_filenames
clean_filenames = []
for file in clean_txt_list:
    clean_filenames.append(file[:-4])

In [ ]:
unmatched_files = []
for file in biblio_filenames:
    if file in clean_filenames:
        print(file, 'matched!')
    else:
        print(file, 'not found in txts!')
        unmatched_files.append(file)
unmatched_files

In [ ]:
filenamefilter = dl_files['Filename'].str.contains('acpq_1990_0064_0002_0233')
dl_files[filenamefilter]

Okay so far, then, i can match up each txt file with its bibliographic information. the next good thing to do would be attach the philpapers identifiers to these for clean linking to philpapers sites.

the final step then would be to put together JSON files that include:
a document for the bibliographic information
a document for all the citation information.

Once I have that I'll be in a position to put the data into the database and then build the website.

Actually, though, I'm going to need to rerun and recalculate all teh searches and comparative data because i've added a couple pieces! That will come last once i'm sure i've got everything else in place

Before I do that I'll go back and edit the class files and function lists to add the work I've done here.

In [ ]:
#Starting to think through creating the Paper with bibliographic information. 
#1. Create the newPaper file
#2. use the matching datafram to pull:
#Author
#Title of work
#Journal
#Volume
#Issue
#Pages
#Year
#Book Title
#3. 
def biblioGenerator(filename, data_frame):
    filename_filter = data_frame['Filename'].str.contains(filename)
    df_row = data_frame[filename_filter]
    do = {}
    do['Author'] = df_row['Author'].values[0]
    do['Title of Work'] = df_row['Title of Work'].values[0]
    do['Journal'] = df_row['Journal'].values[0]
    do['Volume'] = df_row['Volume'].values[0]
    do['Issue'] = df_row['Issue'].values[0]
    do['Pages'] = df_row['Pages'].values[0]
    do['Year'] = df_row['Year'].values[0]
    do['Book Title'] = df_row['Book Title'].values[0]
    
    return do
    
def biblioSetter(paper_obj, data_frame):
    paper_obj.biblio = biblioGenerator(paper_obj.name, data_frame)
    

In [ ]:
cottrell_pap = newPaper('533Cottrell')

In [ ]:
cotpap = newPaper('533Cottrell')

In [ ]:
cotpap.parenthesesCapture()

In [ ]:
biblioSetter(cotpap, dl_files)

In [ ]:
cotpap.biblio

In [ ]:
def final_single_process(filename, data_frame):
    paper = newPaper(filename)
    processPaper(paper)
    paperScoreSetter(paper)
    biblioSetter(paper, data_frame)
    
    return paper

In [ ]:
final_test = final_single_process('533Cottrell', dl_files)

In [ ]:
final_test.biblio

In [ ]:
final_test.totalStrictCites

In [ ]:
final_test.totalAggressiveCites

In [ ]:
pft = p_to_dict(final_test)

In [ ]:
cpft = p_from_dict(pft)

In [ ]:
cpft.name

In [ ]:
for paper in dl_files['Filename'].values:
    print(paper[:-4])

I have a list of files
I have a database to tie those files to bibliographic information
I can search and score these and export them with strict and aggressive chapter and paragraph scores to json format.

That can be a persistent database

Going forward what I need to do is to add a filename and a text copy of the file

In [ ]:
for filename in dl_files['Filename'].values:
    try:
        #process the paper, using dl_files to create bibliographic info
        processed_paper = final_single_process(filename[:-4], dl_files)
        #turn the processed paper into a dictinoary
        paper_dictionary = p_to_dict(processed_paper)
    except:
        print('Error processing', filename[:-4])
    #create the json file
    try:
        json_file_name = filename[:-4]+'.json'
        jfile = open('jsons/'+json_file_name, 'w')
        json.dump(paper_dictionary, jfile)
        jfile.close()
    except:
        print('Error creating JSON file for', filename[:-4])
print('Done creating JSONS')

In [ ]:
jfile = open('jsons/strawson-evident-connexion.json', 'r')
jd = json.load(jfile)
jpaper = p_from_dict(jd)
jfile.close()

In [ ]:
jpaper.s_w_c['1.1.1']

In [ ]:
#okay so i have all the json file data. I can go back and forth between json files and paper data.
#I can use that to generate csvs for the different score sheets.
#I can use those csvs to generate literature level data
#next I need to think through the way I want to store this data on the google cloud firestore.
#for developing the local server though maybe I just use the json load data

#also time to update all the code now in the main function list